<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ParsingWebpages(JARVIS_DFT).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install jarvis-tools

In [ ]:
jid_example="JVASP-1044"


In [ ]:
from jarvis.db.webpages import Webpage
w = Webpage(jid=jid_example)

In [ ]:
data = w.data
print (data['basic_info'].keys())

Here *data* contains the webpage information as nested key-value pairs. Note that these arrays could be in string format, so we convert them into floats before plotting etc.

Only few examples are shown below:

Basic information table

In [ ]:
data['basic_info']['main_relax_info'].keys()

In [ ]:
print ('Formula:',data['basic_info']['main_relax_info']['formula'],', Spacegroup:',data['basic_info']['main_relax_info']['spg_space_group_symbol'])

In [ ]:
from jarvis.io.vasp.inputs import Poscar
pos=Poscar.from_string(data['basic_info']['main_relax_info']['contcar'].replace('\\n','\n').strip('"').strip("'"))
atoms=pos.atoms
print (atoms)

Bandstructure

In [ ]:
print (data['basic_info']['main_band']['main_bands_info'].keys())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
p = data['basic_info']['main_band']['main_bands_info']['spin_up_bands_y'].strip("'").strip('"').split(';')
y=[]
for i in p:
    x =np.array(i.split(','),dtype='float')
    y.append(x)
y=np.array(y).T

In [ ]:
plt.plot(y,'.',c='b')
plt.show()

XRD information

In [ ]:
data["basic_info"]["main_relax_info"]["XRD"].keys()

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
two_theta = np.array(data["basic_info"]["main_relax_info"]["XRD"]['two_thetas'].strip('"').strip("'").split(","),dtype='float')
ints = np.array(data["basic_info"]["main_relax_info"]["XRD"]['intensities'].strip('"').strip("'").split(","),dtype='float')
plt.bar(two_theta,ints)
plt.xlabel('2$\\theta$')
plt.ylabel('Intensity')

Electronic density of states

In [ ]:
data["basic_info"]["main_relax_info"]["main_relax_dos"].keys()

In [ ]:
energies = np.array(data["basic_info"]["main_relax_info"]["main_relax_dos"]['edos_energies'].strip('"').strip("'").split(','),dtype='float')
total_edos_up = np.array(data["basic_info"]["main_relax_info"]["main_relax_dos"]['total_edos_up'].strip('"').strip("'").split(','),dtype='float')
total_edos_dn = np.array(data["basic_info"]["main_relax_info"]["main_relax_dos"]['total_edos_down'].strip('"').strip("'").split(','),dtype='float')

In [ ]:
plt.plot(energies,total_edos_up)
plt.plot(energies,total_edos_dn)
plt.ylabel('E-Ef (eV)')

Frequency dependent dielectric fucntion

In [ ]:
energies = np.array(data["basic_info"]["main_optics_mbj"]["main_optics_mbj_info"]["energies"].strip('"').strip("'").split(","),dtype="float")
real_part_xx = np.array(data["basic_info"]["main_optics_mbj"]["main_optics_mbj_info"]["real_1"].strip('"').strip("'").split(","),dtype="float")
plt.plot(energies,real_part_xx)
plt.xlabel('Energy (eV)')
plt.ylabel('Real $\epsilon_{xx}$')
plt.xlim([0,30])

Electronic bandstructure

In [ ]:
x = data["basic_info"]["main_band"]["main_bands_info"]["spin_up_bands_x"].strip('"').strip("'").split(';')
y = data["basic_info"]["main_band"]["main_bands_info"]["spin_up_bands_y"].strip('"').strip("'").split(';')
kp_labels = data["basic_info"]["main_band"]["main_bands_info"]['kp_labels'].strip('"').strip("'").split(',')
kp_labels_points = np.array(data["basic_info"]["main_band"]["main_bands_info"]['kp_labels_points'].strip('"').strip("'").split(','),dtype='float')

In [ ]:
for i,j in zip(x,y):
  plt.plot(np.array(i.split(','),dtype='float'),np.array(j.split(','),dtype='float'),'.',c='b')
plt.ylabel('E-Ef (eV)')
plt.ylim([-5,1])
plt.xticks(kp_labels_points,kp_labels)
plt.show()

Elastic tensor and associated gamma-point phonons using finite difference method (ISIF = 3, IBRION =6)

In [ ]:
data["basic_info"]["main_elastic"]["main_elastic_info"].keys()

In [ ]:
cij = np.array([i.split(',') for i in data["basic_info"]["main_elastic"]["main_elastic_info"]['cij'].strip('"').strip("'").split(';')],dtype='float')

In [ ]:
from jarvis.analysis.elastic.tensor import ElasticTensor
et = ElasticTensor(et_tensor=cij)

In [ ]:
et.to_dict()

In [ ]:
print ('Debye temp. K:', et.debye_temperature(atoms=atoms))

Raw input/output files

In [ ]:
data['basic_info']['main_relax_info']['download_files'].strip('"').strip("'").split(',')